# **Actividad 1 A2IC — Optimización bajo incertidumbre**
### Unit Commitment estocástico de dos etapas
**EL4114 – Optimización · Departamento de Ingeniería Eléctrica · Universidad de Chile**

La creciente integración de fuentes de energía renovable dependientes del clima ha llevado la incertidumbre a un primer plano en la planificación y operación de los sistemas eléctricos modernos. La generación eólica y solar es intrínsecamente intermitente y difícil de pronosticar con absoluta precisión. Además, los sistemas eléctricos contemporáneos deben lidiar con una serie de incertidumbres a corto y largo plazo, que incluyen la demanda volátil de electricidad, fallas inesperadas de equipos y contingencias en las líneas de transmisión.

No considerar la incertidumbre vuelve al modelo extremadamente frágil y peligrosamente optimista, ya que asume un escenario ideal que rara vez ocurre. Esto provoca que el sistema falle ante el más mínimo cambio de la realidad, llevando a una subestimación del riesgo, malas inversiones (por exceso o falta de capacidad) y decisiones operativas ineficientes que no tienen margen de maniobra para adaptarse a la volatilidad del mundo real.

Este notebook implementa, paso a paso, un problema simplificado de *unit commitment* (UC) con
incertidumbre en **demanda** y **generación solar**:

1. **Parte 1** – Análisis del problema determinista (un único pronóstico).
2. **Parte 2** – Construcción de escenarios.
3. **Parte 3‑4** – Formulación e implementación estocástica de dos etapas.
4. **Parte 5** – Análisis de resultados (determinista vs estocástica, VSS).
5. **Sección 6** – Modelo determinista con reservas conservadoras (enfoque histórico).
6. **Sección 7** – Sample Average Approximation (SAA): muestreo desde distribuciones.
7. **Sección 8** – Oráculo (Perfect Information): evalúa todos los modelos.
8. **Sección 9** – Conclusiones y comparación final.

# 0. Instalación de dependencias
Ejecute esta celda una sola vez (en Colab tarda unos segundos).

In [ ]:
!pip install pulp plotly --quiet

In [ ]:
import numpy as np
import pulp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"   # cambie a "notebook" si usa Jupyter local, o "browser"
print("PuLP", pulp.__version__)

PuLP 3.3.1


# Parte 1 — Análisis del problema determinista

### 1.1 Conjuntos, parámetros y variables

**Conjuntos**
- $\mathcal{G}$: unidades convencionales.
- $\mathcal{T}$: períodos (24 horas).
- $\mathcal{S}$: escenarios.

**Parámetros**
- $P^{\min}_g, P^{\max}_g$: límites de potencia.
- $c_g$: costo variable [\$/MWh].
- $c^{nl}_g$: costo de *no-load* (estar encendida) [\$/h].
- $c^{su}_g$: costo de partida [\$].
- $D_{t,s}$: demanda.
- $P^{sol}_{t,s}$: solar disponible.
- $\text{VOLL}$: penalización por energía no suministrada.

**Variables**
- $u_{g,t}\in\{0,1\}$ encendida
- $v_{g,t}$ partida -
- $w_{g,t}$ apagada  → **primera etapa**
- $p_{g,t,s}\ge0$ despacho convencional · $p^{sol}_{t,s}\ge0$ solar usada · $\ell_{t,s}\ge0$ energía no suministrada → **segunda etapa**


In [ ]:
# ---------------- DATOS DEL SISTEMA ----------------
T = list(range(24))                 # periodos 0..23 (horas)
G = ['G1', 'G2', 'G3']              # unidades convencionales

Pmin   = {'G1': 40,  'G2': 20,  'G3': 10}    # MW
Pmax   = {'G1': 150, 'G2': 100, 'G3': 80}    # MW
cvar   = {'G1': 20,  'G2': 40,  'G3': 70}    # $/MWh  (G1 base barata, G3 peaker cara)
cnl    = {'G1': 100, 'G2': 80,  'G3': 50}    # $/h    costo de estar encendida
cstart = {'G1': 500, 'G2': 300, 'G3': 100}   # $      costo de partida
VOLL   = 10000                               # $/MWh  costo de energia no suministrada

# Perfil horario de demanda (normalizado 0-1) y escala a MW
perfil_D = np.array([0.65,0.60,0.58,0.56,0.58,0.62,0.70,0.80,0.88,0.92,0.95,0.97,
                     0.98,0.96,0.93,0.92,0.94,1.00,0.98,0.95,0.90,0.85,0.78,0.70])
D_base = 270 * perfil_D                       # MW

# Perfil solar: campana diurna (cero de noche, maximo ~mediodia)
Psol_base = np.array([max(0.0, 150*np.sin(np.pi*(h-6)/12)) if 6 <= h <= 18 else 0.0
                      for h in T])

print("Capacidad convencional total:", sum(Pmax.values()), "MW")
print("Demanda maxima (base):", round(D_base.max(),1), "MW   |   Solar maxima (base):", round(Psol_base.max(),1), "MW")

Capacidad convencional total: 330 MW
Demanda maxima (base): 270.0 MW   |   Solar maxima (base): 150.0 MW


In [ ]:
# Visualizacion del pronostico determinista (base)
fig = go.Figure()
fig.add_bar(x=T, y=Psol_base, name="Solar disponible", marker_color="#f4a261")
fig.add_scatter(x=T, y=D_base, name="Demanda", mode="lines+markers",
                line=dict(color="#264653", width=3))
fig.add_hline(y=sum(Pmax.values()), line_dash="dash", line_color="crimson",
              annotation_text="Capacidad convencional total")
fig.update_layout(title="Parte 1 — Pronóstico determinista (escenario base)",
                  xaxis_title="Hora", yaxis_title="MW", template="plotly_white",
                  legend=dict(orientation="h"))
fig.show()

### 1.2 Función objetivo, balance y lógica de encendido

$$\min\; \underbrace{\sum_{g,t}\big(c^{nl}_g\,u_{g,t}+c^{su}_g\,v_{g,t}\big)}_{\text{1ª etapa (compromiso)}}
\;+\; \underbrace{\sum_{s\in\mathcal S}\pi_s\sum_{t}\Big(\sum_g c_g\,p_{g,t,s}+\text{VOLL}\,\ell_{t,s}\Big)}_{\text{2ª etapa (operación esperada)}}$$

**Restricciones** (para cada $t$ y cada $s$):

- Balance: $\sum_g p_{g,t,s} + p^{sol}_{t,s} + \ell_{t,s} = D_{t,s}$
- Límites: $P^{\min}_g\,u_{g,t}\le p_{g,t,s}\le P^{\max}_g\,u_{g,t}$
- Solar: $p^{sol}_{t,s}\le P^{sol}_{t,s}$
- Lógica temporal: $u_{g,t}-u_{g,t-1}=v_{g,t}-w_{g,t}$, con $v_{g,t}+w_{g,t}\le 1$

La función `resolver_uc` de abajo construye exactamente este modelo y sirve **tanto para el caso
determinista (1 escenario) como para el estocástico (varios escenarios)**: solo cambia el diccionario
de escenarios que se le entrega.


In [ ]:
def resolver_uc(D_in, Sol_in, pi_in, fijar_uc=None, nombre="UC"):
    """
    Resuelve el Unit Commitment (de dos etapas si hay >1 escenario).
    D_in, Sol_in : dict {escenario: array de 24 valores}
    pi_in        : dict {escenario: probabilidad}
    fijar_uc     : dict {(g,t): 0/1} para FIJAR el compromiso (usado en EEV). None = libre.
    """
    Sset = list(D_in.keys())
    m = pulp.LpProblem(nombre, pulp.LpMinimize)

    # --- Variables de 1a etapa (NO dependen del escenario) ---
    u = pulp.LpVariable.dicts("u", (G, T), cat="Binary")
    v = pulp.LpVariable.dicts("v", (G, T), cat="Binary")
    w = pulp.LpVariable.dicts("w", (G, T), cat="Binary")
    # --- Variables de 2a etapa (SI dependen del escenario s) ---
    p   = pulp.LpVariable.dicts("p",   (G, T, Sset), lowBound=0)
    ps  = pulp.LpVariable.dicts("ps",  (T, Sset), lowBound=0)
    ell = pulp.LpVariable.dicts("ell", (T, Sset), lowBound=0)

    # --- Funcion objetivo ---
    costo_compromiso = pulp.lpSum(cnl[g]*u[g][t] + cstart[g]*v[g][t] for g in G for t in T)
    costo_operacion  = pulp.lpSum(
        pi_in[s]*(pulp.lpSum(cvar[g]*p[g][t][s] for g in G for t in T)
                  + pulp.lpSum(VOLL*ell[t][s] for t in T))
        for s in Sset)
    m += costo_compromiso + costo_operacion

    # --- Restricciones ---
    for g in G:
        for t in T:
            ut_1 = 0 if t == 0 else u[g][t-1]          # estado inicial: apagada
            m += u[g][t] - ut_1 == v[g][t] - w[g][t]   # logica encendido/apagado
            m += v[g][t] + w[g][t] <= 1
            for s in Sset:
                m += p[g][t][s] >= Pmin[g]*u[g][t]     # minimo tecnico si esta ON
                m += p[g][t][s] <= Pmax[g]*u[g][t]     # maximo / apagada => 0
    for t in T:
        for s in Sset:
            m += ps[t][s] <= Sol_in[s][t]              # solo se usa solar disponible
            m += (pulp.lpSum(p[g][t][s] for g in G)    # BALANCE de potencia
                  + ps[t][s] + ell[t][s] == D_in[s][t])

    if fijar_uc is not None:                           # fijar compromiso (para EEV)
        for g in G:
            for t in T:
                m += u[g][t] == fijar_uc[(g, t)]

    m.solve(pulp.PULP_CBC_CMD(msg=0))

    return dict(
        status=pulp.LpStatus[m.status],
        obj=pulp.value(m.objective),
        u={(g, t): round(u[g][t].value())   for g in G for t in T},
        v={(g, t): round(v[g][t].value())   for g in G for t in T},
        w={(g, t): round(w[g][t].value())   for g in G for t in T},
        p={(g, t, s): p[g][t][s].value()    for g in G for t in T for s in Sset},
        ps={(t, s): ps[t][s].value()        for t in T for s in Sset},
        ell={(t, s): ell[t][s].value()      for t in T for s in Sset},
        Sset=Sset)

In [ ]:
# --- Resolver el caso DETERMINISTA: un unico escenario "det" con prob 1 ---
det = resolver_uc({"det": D_base}, {"det": Psol_base}, {"det": 1.0}, nombre="Determinista")
print("Estado:", det["status"])
print("Costo total determinista: $", round(det["obj"], 1))

# Tabla de encendido (u) por unidad y hora
print("\nCompromiso de unidades (1 = encendida):")
print("Hora:", "".join(f"{t:>3}" for t in T))
for g in G:
    print(f"{g}:  ", "".join(f"{det['u'][(g,t)]:>3}" for t in T))

Estado: Optimal
Costo total determinista: $ 105687.8

Compromiso de unidades (1 = encendida):
Hora:   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
G1:     1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
G2:     1  1  1  1  1  1  1  1  1  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1
G3:     0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  0  0  0  0


In [ ]:
def plot_despacho(res, s, D_s, titulo):
    """Grafico de area apilada: aporte de cada unidad + solar + ENS vs demanda."""
    fig = go.Figure()
    colores = {"G1": "#2a9d8f", "G2": "#e9c46a", "G3": "#e76f51"}
    for g in G:
        fig.add_scatter(x=T, y=[res["p"][(g, t, s)] for t in T], name=g,
                        stackgroup="one", mode="none", fillcolor=colores[g])
    fig.add_scatter(x=T, y=[res["ps"][(t, s)] for t in T], name="Solar usada",
                    stackgroup="one", mode="none", fillcolor="#f4a261")
    fig.add_scatter(x=T, y=[res["ell"][(t, s)] for t in T], name="No suministrada",
                    stackgroup="one", mode="none", fillcolor="#9b2226")
    fig.add_scatter(x=T, y=D_s, name="Demanda", mode="lines+markers",
                    line=dict(color="black", width=2, dash="dot"))
    fig.update_layout(title=titulo, xaxis_title="Hora", yaxis_title="MW",
                      template="plotly_white", legend=dict(orientation="h"))
    fig.show()

plot_despacho(det, "det", D_base, "Parte 1 — Despacho determinista")

## Parte 2 — Construcción de escenarios

Incorporamos incertidumbre con **3 escenarios** que representan trayectorias alternativas del sistema.
Cada uno escala los perfiles base de demanda y solar y tiene una probabilidad $\pi_s$:

| Escenario | Demanda | Solar | $\pi_s$ | Interpretación |
|-----------|---------|-------|---------|----------------|
| s1 | ×0.90 | ×1.10 | 0.25 | Día soleado, demanda baja |
| s2 | ×1.00 | ×1.00 | 0.50 | Caso base |
| s3 | ×1.25 | ×0.40 | 0.25 | Día nublado, demanda alta (estrés) |

> El escenario **s3** es el crítico: junta alta demanda con poca solar, por lo que exige más
> generación convencional. El compromiso de primera etapa debe **cubrirse (hedge)** ante esta posibilidad.


In [ ]:
# Definicion de escenarios
escenarios = {
    "s1": dict(nombre="Soleado / baja demanda", kd=0.90, ks=1.10, pi=0.25),
    "s2": dict(nombre="Base",                   kd=1.00, ks=1.00, pi=0.50),
    "s3": dict(nombre="Nublado / alta demanda", kd=1.25, ks=0.40, pi=0.25),
}
S   = list(escenarios.keys())
D   = {s: escenarios[s]["kd"] * D_base    for s in S}   # demanda por escenario
Sol = {s: escenarios[s]["ks"] * Psol_base for s in S}   # solar por escenario
pi  = {s: escenarios[s]["pi"]             for s in S}
assert abs(sum(pi.values()) - 1) < 1e-9, "Las probabilidades deben sumar 1"
print("Probabilidades:", pi, "  suma =", sum(pi.values()))

Probabilidades: {'s1': 0.25, 's2': 0.5, 's3': 0.25}   suma = 1.0


In [ ]:
# Visualizacion interactiva de los escenarios
fig = make_subplots(rows=1, cols=2, subplot_titles=("Demanda por escenario", "Solar disponible por escenario"))
col = {"s1": "#2a9d8f", "s2": "#264653", "s3": "#e76f51"}
for s in S:
    nom = f"{s}: {escenarios[s]['nombre']}"
    fig.add_scatter(x=T, y=D[s],   name=nom, line=dict(color=col[s]), row=1, col=1, legendgroup=s)
    fig.add_scatter(x=T, y=Sol[s], name=nom, line=dict(color=col[s]), row=1, col=2,
                    legendgroup=s, showlegend=False)
fig.update_layout(title="Parte 2 — Escenarios de demanda y generación solar",
                  template="plotly_white", legend=dict(orientation="h"), height=420)
fig.update_xaxes(title_text="Hora"); fig.update_yaxes(title_text="MW")
fig.show()

## Partes 3 y 4 — Formulación estocástica e implementación

Resolvemos el **equivalente determinista** de la ecuación (2) del enunciado:

$$\min_{x}\; c^\top x + \sum_{s\in\mathcal S}\pi_s\,Q(x,\xi_s)$$

usando la misma función `resolver_uc`, ahora con los **3 escenarios**. Verificamos los 5 puntos
exigidos en la Parte 4:

1. Las binarias $u,v,w$ **no** llevan índice $s$ ✔ (definidas sobre `(G, T)`).
2. Las de despacho $p, p^{sol}, \ell$ **sí** llevan índice $s$ ✔ (definidas sobre `(..., Sset)`).
3. El objetivo usa el **costo esperado** $\sum_s\pi_s(\cdot)$ ✔.
4. El balance se impone **por período y escenario** ✔ (doble bucle `t, s`).
5. Demanda y solar están **indexadas por escenario** ✔ (`D_in[s][t]`, `Sol_in[s][t]`).


In [ ]:
# --- Resolver el modelo ESTOCASTICO (Recourse Problem, RP) ---
rp = resolver_uc(D, Sol, pi, nombre="Estocastico_RP")
print("Estado:", rp["status"])
print("Costo esperado (RP): $", round(rp["obj"], 1))

Estado: Optimal
Costo esperado (RP): $ 133414.5


In [ ]:
# Mapa de calor del compromiso de unidades (primera etapa, comun a todos los escenarios)
M = np.array([[rp["u"][(g, t)] for t in T] for g in G])
fig = go.Figure(go.Heatmap(z=M, x=[f"h{t}" for t in T], y=G,
                           colorscale=[[0, "#e9ecef"], [1, "#2a9d8f"]],
                           showscale=False, xgap=1, ygap=3))
for i, g in enumerate(G):
    for t in T:
        if rp["u"][(g, t)] == 1:
            fig.add_annotation(x=f"h{t}", y=g, text="ON", showarrow=False,
                               font=dict(size=9, color="white"))
fig.update_layout(title="Partes 3-4 — Compromiso de unidades (1ª etapa, no depende de s)",
                  template="plotly_white", height=300)
fig.show()

In [ ]:
# Despacho de 2a etapa para CADA escenario (subplots de area apilada)
fig = make_subplots(rows=1, cols=len(S),
                    subplot_titles=[f"{s}: {escenarios[s]['nombre']}" for s in S])
colores = {"G1": "#2a9d8f", "G2": "#e9c46a", "G3": "#e76f51"}
for j, s in enumerate(S, start=1):
    showleg = (j == 1)
    for g in G:
        fig.add_scatter(x=T, y=[rp["p"][(g, t, s)] for t in T], name=g, legendgroup=g,
                        stackgroup=s, mode="none", fillcolor=colores[g],
                        showlegend=showleg, row=1, col=j)
    fig.add_scatter(x=T, y=[rp["ps"][(t, s)] for t in T], name="Solar", legendgroup="Solar",
                    stackgroup=s, mode="none", fillcolor="#f4a261", showlegend=showleg, row=1, col=j)
    fig.add_scatter(x=T, y=[rp["ell"][(t, s)] for t in T], name="No suministrada",
                    legendgroup="ENS", stackgroup=s, mode="none", fillcolor="#9b2226",
                    showlegend=showleg, row=1, col=j)
    fig.add_scatter(x=T, y=D[s], name="Demanda", legendgroup="D", mode="lines",
                    line=dict(color="black", width=1.5, dash="dot"),
                    showlegend=showleg, row=1, col=j)
fig.update_layout(title="Partes 3-4 — Despacho de 2ª etapa por escenario",
                  template="plotly_white", height=430, legend=dict(orientation="h"))
fig.update_xaxes(title_text="Hora"); fig.update_yaxes(title_text="MW")
fig.show()

## Parte 5 — Análisis de resultados

Respondemos las preguntas orientadoras: energía no suministrada por escenario, impacto de la
incertidumbre solar y, sobre todo, la **comparación entre la solución estocástica y la determinista
basada en valores esperados** mediante el **Valor de la Solución Estocástica (VSS)**.

- **RP** (*Recourse Problem*): costo esperado del modelo estocástico = solución de referencia.
- **EV** (*Expected Value problem*): se resuelve el determinista usando $\bar D_t=\sum_s\pi_s D_{t,s}$ y $\bar P^{sol}_t$.
- **EEV** (*Expected result of the EV solution*): se **fija** el compromiso del EV y se evalúa su costo esperado sobre los escenarios reales.
- **VSS** $= \text{EEV} - \text{RP} \ge 0$: cuánto se ahorra por planificar con incertidumbre en vez de con promedios.


In [ ]:
# Energia no suministrada (ENS) por escenario
print("Energía no suministrada (MWh) por escenario:")
for s in S:
    tot = sum(rp["ell"][(t, s)] for t in T)
    horas = [t for t in T if rp["ell"][(t, s)] > 1e-4]
    print(f"  {s} ({escenarios[s]['nombre']}): {tot:6.2f} MWh   horas con ENS: {horas}")

Energía no suministrada (MWh) por escenario:
  s1 (Soleado / baja demanda):   0.00 MWh   horas con ENS: []
  s2 (Base):   0.00 MWh   horas con ENS: []
  s3 (Nublado / alta demanda):   0.75 MWh   horas con ENS: [18]


In [ ]:
# Comparacion ESTOCASTICA vs DETERMINISTA-CON-PROMEDIOS (VSS)
D_ev   = {"ev": sum(pi[s] * D[s]   for s in S)}    # demanda esperada
Sol_ev = {"ev": sum(pi[s] * Sol[s] for s in S)}    # solar esperada

ev  = resolver_uc(D_ev, Sol_ev, {"ev": 1.0}, nombre="EV")
eev = resolver_uc(D, Sol, pi, fijar_uc={k: ev["u"][k] for k in ev["u"]}, nombre="EEV")
vss = eev["obj"] - rp["obj"]

print(f"RP  (estocástico)            : $ {rp['obj']:.1f}")
print(f"EV  (determinista promedios) : $ {ev['obj']:.1f}   <- subestima el costo real")
print(f"EEV (decisión EV evaluada)   : $ {eev['obj']:.1f}")
print(f"VSS = EEV - RP               : $ {vss:.1f}")

RP  (estocástico)            : $ 133414.5
EV  (determinista promedios) : $ 117342.7   <- subestima el costo real
EEV (decisión EV evaluada)   : $ 1877309.0
VSS = EEV - RP               : $ 1743894.5


In [ ]:
# Grafico comparativo de costos
fig = go.Figure(go.Bar(
    x=["EV<br>(promedios)", "RP<br>(estocástico)", "EEV<br>(decisión EV<br>en escenarios reales)"],
    y=[ev["obj"], rp["obj"], eev["obj"]],
    marker_color=["#94a3b8", "#2a9d8f", "#9b2226"],
    text=[f"${ev['obj']:,.0f}", f"${rp['obj']:,.0f}", f"${eev['obj']:,.0f}"],
    textposition="outside"))
fig.add_annotation(x="EEV<br>(decisión EV<br>en escenarios reales)", y=eev["obj"],
                   text=f"VSS = ${vss:,.0f}", showarrow=True, ay=-40, font=dict(color="#9b2226"))
fig.update_layout(title="Parte 5 — Costos: el VSS mide el valor de planificar con incertidumbre",
                  yaxis_title="Costo total [$]", template="plotly_white", height=430)
fig.show()

## Sección 6 — UC determinista con reservas conservadoras

Históricamente, la planificación de sistemas eléctricos maneja la incertidumbre con **requisitos de reserva
conservadores**: se resuelve un modelo determinista (un solo pronóstico) pero se exige que la capacidad
comprometida supere la demanda por un margen fijo:

$$\sum_g P^{\max}_g \cdot u_{g,t} \geq (1 + \alpha) \cdot D_t \quad \forall\, t$$

donde $\alpha$ es el margen de reserva (p.ej. 15%). Este enfoque:
- ✅ Es computacionalmente sencillo (un solo escenario)
- ❌ Es **económicamente ineficiente** (sobrecompromete unidades caras "por si acaso")
- ❌ **No modela** la distribución de la incertidumbre
- ❌ **No diferencia** escenarios probables de improbables

Resolveremos el UC determinista con distintos márgenes $\alpha \in \{0\%,\,10\%,\,15\%,\,20\%\}$
para observar el trade-off costo vs seguridad.

In [ ]:
def resolver_uc_reserva(D_nom, Sol_nom, reserva=0.15, nombre="UC_Reserva"):
    """
    UC determinista con restriccion de reserva de capacidad.
    D_nom, Sol_nom : arrays de 24 valores (un solo pronostico).
    reserva        : fraccion de reserva (0.15 = 15%).
    """
    m = pulp.LpProblem(nombre, pulp.LpMinimize)
    u = pulp.LpVariable.dicts("u", (G, T), cat="Binary")
    v = pulp.LpVariable.dicts("v", (G, T), cat="Binary")
    w = pulp.LpVariable.dicts("w", (G, T), cat="Binary")
    p = pulp.LpVariable.dicts("p", (G, T), lowBound=0)
    ps = pulp.LpVariable.dicts("ps", T, lowBound=0)
    ell = pulp.LpVariable.dicts("ell", T, lowBound=0)

    # Objetivo: compromiso + operacion determinista
    m += (pulp.lpSum(cnl[g]*u[g][t] + cstart[g]*v[g][t] for g in G for t in T)
          + pulp.lpSum(cvar[g]*p[g][t] for g in G for t in T)
          + pulp.lpSum(VOLL*ell[t] for t in T))

    for g in G:
        for t in T:
            ut_1 = 0 if t == 0 else u[g][t-1]
            m += u[g][t] - ut_1 == v[g][t] - w[g][t]
            m += v[g][t] + w[g][t] <= 1
            m += p[g][t] >= Pmin[g]*u[g][t]
            m += p[g][t] <= Pmax[g]*u[g][t]
    for t in T:
        m += ps[t] <= Sol_nom[t]
        m += pulp.lpSum(p[g][t] for g in G) + ps[t] + ell[t] == D_nom[t]
        # RESTRICCION DE RESERVA
        m += pulp.lpSum(Pmax[g]*u[g][t] for g in G) >= (1 + reserva) * D_nom[t]

    m.solve(pulp.PULP_CBC_CMD(msg=0))

    return dict(
        status=pulp.LpStatus[m.status],
        obj=pulp.value(m.objective),
        u={(g, t): round(u[g][t].value()) for g in G for t in T},
        v={(g, t): round(v[g][t].value()) for g in G for t in T},
        w={(g, t): round(w[g][t].value()) for g in G for t in T},
        p={(g, t): p[g][t].value() for g in G for t in T},
        ps={t: ps[t].value() for t in T},
        ell={t: ell[t].value() for t in T},
    )

In [ ]:
# Resolver con distintos margenes de reserva
alphas = [0.0, 0.10, 0.15, 0.20]
resultados_reserva = {}
for alpha in alphas:
    res = resolver_uc_reserva(D_base, Psol_base, reserva=alpha,
                              nombre=f"Reserva_{int(alpha*100)}pct")
    resultados_reserva[alpha] = res
    n_on = sum(res["u"][(g, t)] for g in G for t in T)
    ens_tot = sum(res["ell"][t] for t in T)
    print(f"Reserva {alpha*100:4.0f}%: costo=${res['obj']:>10,.1f}  "
          f"unidades-hora ON={n_on:>3.0f}  ENS={ens_tot:.1f} MWh  [{res['status']}]")

# Heatmaps de compromiso: reserva 0% vs 20%
fig = make_subplots(rows=1, cols=2, subplot_titles=["Reserva 0% (sin margen)", "Reserva 20% (conservador)"])
for j, alpha in enumerate([0.0, 0.20]):
    res = resultados_reserva[alpha]
    M = np.array([[res["u"][(g, t)] for t in T] for g in G])
    fig.add_trace(go.Heatmap(z=M, x=[f"h{t}" for t in T], y=G,
                             colorscale=[[0, "#e9ecef"], [1, "#e76f51"]],
                             showscale=False, xgap=1, ygap=3), row=1, col=j+1)
fig.update_layout(title="Sección 6 — Compromiso de unidades: sin reserva vs conservador",
                  template="plotly_white", height=280)
fig.show()

# Grafico de barras: costo por nivel de reserva
fig = go.Figure(go.Bar(
    x=[f"{a*100:.0f}%" for a in alphas],
    y=[resultados_reserva[a]["obj"] for a in alphas],
    marker_color=["#94a3b8", "#e9c46a", "#e76f51", "#9b2226"],
    text=[f"${resultados_reserva[a]['obj']:,.0f}" for a in alphas],
    textposition="outside"))
fig.update_layout(title="Sección 6 — Trade-off: costo total vs margen de reserva",
                  xaxis_title="Margen de reserva (α)", yaxis_title="Costo total [$]",
                  template="plotly_white", height=400)
fig.show()

Reserva    0%: costo=$ 112,321.8  unidades-hora ON= 57  ENS=0.0 MWh  [Optimal]
Reserva   10%: costo=$ 114,571.8  unidades-hora ON= 62  ENS=0.0 MWh  [Optimal]
Reserva   15%: costo=$ 114,571.8  unidades-hora ON= 62  ENS=0.0 MWh  [Optimal]
Reserva   20%: costo=$ 115,328.3  unidades-hora ON= 64  ENS=0.0 MWh  [Optimal]


## Sección 7 — Incertidumbre basada en distribuciones (SAA)

En vez de construir escenarios manualmente, modelamos la incertidumbre con **distribuciones de probabilidad**:

- Factor de demanda: $k_D \sim \mathcal{N}(1.0,\; 0.20^2)$
- Factor solar: $k_S \sim \mathcal{N}(1.0,\; 0.50^2)$ (mayor incertidumbre en energía solar)
- Correlación negativa: $\rho = -0.5$ (día nublado → menos solar, más demanda)

La **Sample Average Approximation (SAA)** consiste en:
1. Muestrear $N$ realizaciones $(k_D^{(i)}, k_S^{(i)})$ desde la distribución conjunta
2. Construir escenarios: $D_t^{(i)} = k_D^{(i)} \cdot D_t^{base}$, $\;P_t^{sol,(i)} = k_S^{(i)} \cdot P_t^{sol,base}$
3. Resolver el UC estocástico con estos $N$ escenarios equiprobables ($\pi_i = 1/N$)

**Pregunta clave:** ¿cuántos muestreos $N$ son suficientes para obtener una buena solución?

In [ ]:
def generar_escenarios_SAA(N, seed=42):
    """Muestrea N escenarios de (kD, kS) desde distribucion normal bivariada."""
    rng = np.random.default_rng(seed)
    mu = [1.0, 1.0]
    sigma_D, sigma_S, rho = 0.20, 0.50, -0.5
    cov = [[sigma_D**2, rho*sigma_D*sigma_S],
           [rho*sigma_D*sigma_S, sigma_S**2]]
    samples = rng.multivariate_normal(mu, cov, size=N)
    samples[:, 0] = np.clip(samples[:, 0], 0.5, 1.6)   # limites fisicos demanda
    samples[:, 1] = np.clip(samples[:, 1], 0.0, 2.0)   # limites fisicos solar

    D_saa, Sol_saa, pi_saa = {}, {}, {}
    for i in range(N):
        s = f"saa_{i}"
        D_saa[s] = samples[i, 0] * D_base
        Sol_saa[s] = samples[i, 1] * Psol_base
        pi_saa[s] = 1.0 / N
    return D_saa, Sol_saa, pi_saa, samples

In [ ]:
# Visualizacion de escenarios muestreados (N=100)
_, _, _, muestras_100 = generar_escenarios_SAA(100, seed=42)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Muestras en espacio (kD, kS)",
                                    "Perfiles de demanda muestreados"])

# Scatter de factores
fig.add_trace(go.Scatter(x=muestras_100[:, 0], y=muestras_100[:, 1], mode="markers",
                         marker=dict(size=6, color="#2a9d8f", opacity=0.7),
                         name="Muestras"), row=1, col=1)
fig.add_trace(go.Scatter(x=[1.0], y=[1.0], mode="markers",
                         marker=dict(size=12, color="red", symbol="x"),
                         name="Base (1,1)"), row=1, col=1)

# Spaghetti de perfiles de demanda
D_100, _, _, _ = generar_escenarios_SAA(100, seed=42)
for i, s in enumerate(D_100):
    fig.add_trace(go.Scatter(x=T, y=D_100[s], mode="lines",
                             line=dict(color="#264653", width=0.5), opacity=0.2,
                             showlegend=(i == 0), name="Escenarios",
                             legendgroup="esc"), row=1, col=2)
fig.add_trace(go.Scatter(x=T, y=D_base, mode="lines",
                         line=dict(color="red", width=2.5, dash="dash"),
                         name="Base"), row=1, col=2)

fig.update_xaxes(title_text="kD (factor demanda)", row=1, col=1)
fig.update_yaxes(title_text="kS (factor solar)", row=1, col=1)
fig.update_xaxes(title_text="Hora", row=1, col=2)
fig.update_yaxes(title_text="MW", row=1, col=2)
fig.update_layout(title="Sección 7 — Escenarios muestreados desde distribución bivariada (N=100)",
                  template="plotly_white", height=420, legend=dict(orientation="h"))
fig.show()

In [ ]:
# Resolver SAA con distintos N
Ns = [2, 5, 10, 25, 50, 100]
resultados_saa = {}

print("Resolviendo SAA para distintos N...")
for N in Ns:
    D_n, Sol_n, pi_n, muestras = generar_escenarios_SAA(N, seed=42)
    saa = resolver_uc(D_n, Sol_n, pi_n, nombre=f"SAA_{N}")
    resultados_saa[N] = {"res": saa, "muestras": muestras}
    n_on = sum(saa["u"][(g, t)] for g in G for t in T)
    print(f"  SAA N={N:>3}: costo in-sample=${saa['obj']:>10,.1f}  "
          f"unidades-hora ON={n_on:>3.0f}  [{saa['status']}]")

Resolviendo SAA para distintos N...
  SAA N=  2: costo in-sample=$  90,507.4  unidades-hora ON= 43  [Optimal]
  SAA N=  5: costo in-sample=$ 109,056.8  unidades-hora ON= 58  [Optimal]
  SAA N= 10: costo in-sample=$ 105,923.3  unidades-hora ON= 59  [Optimal]
  SAA N= 25: costo in-sample=$ 108,919.6  unidades-hora ON= 60  [Optimal]
  SAA N= 50: costo in-sample=$ 122,615.3  unidades-hora ON= 63  [Optimal]
  SAA N=100: costo in-sample=$ 146,960.4  unidades-hora ON= 66  [Optimal]


In [ ]:
# Comparacion visual del compromiso: SAA N=2 vs SAA N=100
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["SAA N=2 (pocos muestreos)",
                                    "SAA N=100 (muchos muestreos)"])
for j, N in enumerate([2, 100]):
    res = resultados_saa[N]["res"]
    M = np.array([[res["u"][(g, t)] for t in T] for g in G])
    fig.add_trace(go.Heatmap(z=M, x=[f"h{t}" for t in T], y=G,
                             colorscale=[[0, "#e9ecef"], [1, "#264653"]],
                             showscale=False, xgap=1, ygap=3), row=1, col=j+1)
fig.update_layout(title="Sección 7 — Compromiso de unidades: pocos vs muchos muestreos",
                  template="plotly_white", height=280)
fig.show()

## Sección 8 — Oráculo (Perfect Information) evalúa todos los modelos

El **oráculo** (o “dios”) conoce el futuro con certeza total. Para cada escenario posible,
resuelve el UC sabiendo exactamente la demanda y solar → obtiene el **costo mínimo posible**
(Wait-and-See, WS).

**Metodología de evaluación out-of-sample (OOS):**
1. Generar $N_{eval} = 200$ escenarios "verdaderos" desde la distribución
2. **Oráculo**: resolver un UC individual por escenario → cota inferior inalcanzable
3. **Cada modelo**: fijar su compromiso $u_{g,t}$ y re-optimizar el despacho sobre los $N_{eval}$ escenarios
4. **Comparar**: $\text{gap}(\%) = \frac{\text{costo modelo} - \text{costo oráculo}}{\text{costo oráculo}} \times 100$

Los modelos evaluados son:
- Reservas (0%, 10%, 15%, 20%)
- Determinista con promedios (EV)
- Estocástico con 3 escenarios manuales (RP)
- SAA con $N \in \{2, 5, 10, 25, 50, 100\}$

In [ ]:
# Generar escenarios de evaluacion out-of-sample
N_eval = 200
D_eval, Sol_eval, pi_eval, _ = generar_escenarios_SAA(N_eval, seed=9999)

# Oraculo: resolver UC por escenario individual (Wait-and-See)
print(f"Calculando oraculo para {N_eval} escenarios...")
ws_costos = {}
for i, s in enumerate(D_eval):
    res_s = resolver_uc({s: D_eval[s]}, {s: Sol_eval[s]}, {s: 1.0}, nombre=f"WS_{s}")
    ws_costos[s] = res_s["obj"]
    if (i+1) % 50 == 0:
        print(f"  ...{i+1}/{N_eval} escenarios resueltos")

WS = sum(pi_eval[s] * ws_costos[s] for s in D_eval)
print(f"\nCosto oraculo (WS) = ${WS:,.1f}")

Calculando oraculo para 200 escenarios...
  ...50/200 escenarios resueltos
  ...100/200 escenarios resueltos
  ...150/200 escenarios resueltos
  ...200/200 escenarios resueltos

Costo oraculo (WS) = $216,183.8


In [ ]:
# Evaluar TODOS los modelos fijando su compromiso sobre escenarios OOS
modelos_a_evaluar = {}

# Modelos de reserva
for alpha in alphas:
    nombre = f"Reserva {alpha*100:.0f}%"
    modelos_a_evaluar[nombre] = resultados_reserva[alpha]["u"]

# Determinista con promedios (EV)
modelos_a_evaluar["EV (promedios)"] = ev["u"]

# Estocastico 3 escenarios manuales (RP)
modelos_a_evaluar["RP (3 esc.)"] = rp["u"]

# SAA con distintos N
for N in Ns:
    modelos_a_evaluar[f"SAA N={N}"] = resultados_saa[N]["res"]["u"]

# Evaluar cada modelo
evaluacion = {}
print("Evaluando modelos en escenarios OOS...")
for nombre, uc_fijo in modelos_a_evaluar.items():
    oos = resolver_uc(D_eval, Sol_eval, pi_eval, fijar_uc=uc_fijo, nombre="OOS")
    ens_total = sum(oos["ell"][(t, s)] for t in T for s in oos["Sset"]) / N_eval
    evaluacion[nombre] = {
        "costo_oos": oos["obj"],
        "ens_esperada": ens_total,
        "gap_pct": (oos["obj"] - WS) / WS * 100
    }
    print(f"  {nombre:>20s}: costo=${oos['obj']:>10,.1f}  "
          f"ENS={ens_total:>6.1f} MWh  gap={evaluacion[nombre]['gap_pct']:.2f}%")

print(f"\n  {'Oraculo (WS)':>20s}: costo=${WS:>10,.1f}  "
      f"ENS=  0.0 MWh  gap=0.00%")

Evaluando modelos en escenarios OOS...
            Reserva 0%: costo=$ 717,933.1  ENS=  59.5 MWh  gap=232.09%
           Reserva 10%: costo=$ 319,151.7  ENS=  19.2 MWh  gap=47.63%
           Reserva 15%: costo=$ 319,151.7  ENS=  19.2 MWh  gap=47.63%
           Reserva 20%: costo=$ 252,032.8  ENS=  12.3 MWh  gap=16.58%
        EV (promedios): costo=$2,042,002.7  ENS= 193.0 MWh  gap=844.57%
           RP (3 esc.): costo=$ 252,032.8  ENS=  12.3 MWh  gap=16.58%
               SAA N=2: costo=$3,277,816.2  ENS= 317.2 MWh  gap=1416.22%
               SAA N=5: costo=$ 548,206.6  ENS=  42.4 MWh  gap=153.58%
              SAA N=10: costo=$ 439,601.8  ENS=  31.4 MWh  gap=103.35%
              SAA N=25: costo=$ 394,617.1  ENS=  26.9 MWh  gap=82.54%
              SAA N=50: costo=$ 274,167.0  ENS=  14.6 MWh  gap=26.82%
             SAA N=100: costo=$ 231,887.5  ENS=  10.2 MWh  gap=7.26%

          Oraculo (WS): costo=$ 216,183.8  ENS=  0.0 MWh  gap=0.00%


In [ ]:
# Grafico de convergencia SAA
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Costo OOS vs N muestras SAA",
                                    "ENS esperada vs N muestras SAA"])

costos_saa = [evaluacion[f"SAA N={N}"]["costo_oos"] for N in Ns]
ens_saa = [evaluacion[f"SAA N={N}"]["ens_esperada"] for N in Ns]

# Costo OOS
fig.add_trace(go.Scatter(x=Ns, y=costos_saa, mode="lines+markers", name="SAA",
                         marker=dict(size=10, color="#264653"),
                         line=dict(width=2.5)), row=1, col=1)
fig.add_hline(y=WS, line_dash="dash", line_color="green", row=1, col=1,
              annotation_text="Oráculo (WS)", annotation_position="bottom right")
fig.add_hline(y=evaluacion["RP (3 esc.)"]["costo_oos"],
              line_dash="dot", line_color="#2a9d8f", row=1, col=1,
              annotation_text="RP (3 esc.)", annotation_position="top right")
fig.add_hline(y=evaluacion["EV (promedios)"]["costo_oos"],
              line_dash="dashdot", line_color="#9b2226", row=1, col=1,
              annotation_text="EV (promedios)", annotation_position="top left")

# ENS
fig.add_trace(go.Scatter(x=Ns, y=ens_saa, mode="lines+markers", name="SAA (ENS)",
                         marker=dict(size=10, color="#e76f51"),
                         line=dict(width=2.5), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="N muestras SAA", type="log", row=1, col=1)
fig.update_xaxes(title_text="N muestras SAA", type="log", row=1, col=2)
fig.update_yaxes(title_text="Costo OOS [$]", row=1, col=1)
fig.update_yaxes(title_text="ENS esperada [MWh]", row=1, col=2)
fig.update_layout(title="Sección 8 — Convergencia SAA: pocos muestreos → solución inadecuada",
                  template="plotly_white", height=450, legend=dict(orientation="h"))
fig.show()

In [ ]:
# Tabla resumen de todos los modelos
print("=" * 80)
print(f"{'Modelo':>25s} {'Costo OOS ($)':>14s} {'ENS (MWh)':>10s} {'Gap vs Oraculo':>15s}")
print("=" * 80)
print(f"{'Oraculo (WS)':>25s} {WS:>14,.1f} {'0.0':>10s} {'0.00%':>15s}")
print("-" * 80)
for nombre, vals in sorted(evaluacion.items(), key=lambda x: x[1]["costo_oos"]):
    print(f"{nombre:>25s} {vals['costo_oos']:>14,.1f} {vals['ens_esperada']:>10.1f} "
          f"{vals['gap_pct']:>14.2f}%")
print("=" * 80)

# Grafico de barras horizontal: todos los modelos ordenados
nombres_ord = sorted(evaluacion.keys(), key=lambda x: evaluacion[x]["costo_oos"])
nombres_plot = ["Oráculo (WS)"] + nombres_ord
costos_plot = [WS] + [evaluacion[n]["costo_oos"] for n in nombres_ord]

colores_plot = ["#2ecc71"]   # oracle = green
for n in nombres_ord:
    if "Reserva" in n:
        colores_plot.append("#e76f51")
    elif "EV" in n:
        colores_plot.append("#94a3b8")
    elif "RP" in n:
        colores_plot.append("#2a9d8f")
    elif "SAA" in n:
        colores_plot.append("#264653")

fig = go.Figure(go.Bar(
    y=nombres_plot, x=costos_plot, orientation="h",
    marker_color=colores_plot,
    text=[f"${c:,.0f}" for c in costos_plot],
    textposition="outside"))
fig.add_vline(x=WS, line_dash="dash", line_color="green", line_width=2,
              annotation_text="Oráculo", annotation_position="top right")
fig.update_layout(title="Sección 8 — Comparación de todos los modelos (evaluación OOS)",
                  xaxis_title="Costo esperado OOS [$]",
                  template="plotly_white", height=550,
                  margin=dict(l=200))
fig.show()

                   Modelo  Costo OOS ($)  ENS (MWh)  Gap vs Oraculo
             Oraculo (WS)      216,183.8        0.0           0.00%
--------------------------------------------------------------------------------
                SAA N=100      231,887.5       10.2           7.26%
              Reserva 20%      252,032.8       12.3          16.58%
              RP (3 esc.)      252,032.8       12.3          16.58%
                 SAA N=50      274,167.0       14.6          26.82%
              Reserva 10%      319,151.7       19.2          47.63%
              Reserva 15%      319,151.7       19.2          47.63%
                 SAA N=25      394,617.1       26.9          82.54%
                 SAA N=10      439,601.8       31.4         103.35%
                  SAA N=5      548,206.6       42.4         153.58%
               Reserva 0%      717,933.1       59.5         232.09%
           EV (promedios)    2,042,002.7      193.0         844.57%
                  SAA N=2    3,277,

## Sección 9 — Conclusiones y comparación final

### Jerarquía de costos esperada

$$\text{WS (oráculo)} \;\leq\; \text{RP / SAA}_{N\to\infty} \;\leq\; \text{EEV} \;\leq\; \text{Reserva alta}$$

### Hallazgos principales

1. **Oráculo (Wait-and-See)**: establece la **cota inferior** inalcanzable. Si pudiéramos predecir
   el futuro perfectamente, este sería el costo mínimo. El **EVPI** (Expected Value of Perfect
   Information) cuantifica el valor de esa información.

2. **Reservas conservadoras**: simples pero ineficientes — sobrecomprometen unidades caras sin
   discriminar entre escenarios probables e improbables. A mayor margen, más seguridad pero
   mayor costo. Además, la reserva se calcula sobre el pronóstico nominal, por lo que no protege
   adecuadamente contra escenarios extremos que exceden el margen.

3. **Determinista (EV)**: subestima el costo real y produce decisiones que fallan ante
   escenarios adversos (alta ENS).

4. **Pocos muestreos SAA (N=2, 5)**: generan soluciones **inadecuadas** — la primera etapa no
   "cubre" suficientes realizaciones de la incertidumbre, resultando en alta ENS y costos
   OOS elevados. La solución depende fuertemente de qué muestras cayeron.

5. **SAA con muchos muestreos (N≥50)**: converge hacia una buena solución, comparable o
   mejor que el RP con escenarios manuales, porque aproxima bien la distribución verdadera.

6. **RP con escenarios bien diseñados**: puede ser tan bueno como SAA con muchas muestras,
   porque los escenarios manuales capturan deliberadamente los casos extremos (ventaja del
   diseño experto).

> **Mensaje clave:** modelar explícitamente la incertidumbre (ya sea con escenarios manuales o SAA
> con suficientes muestras) produce decisiones significativamente mejores que el enfoque determinista
> con reservas. El oráculo muestra que siempre hay un "precio de la incertidumbre" inevitable, pero
> las herramientas estocásticas permiten acercarse mucho más al óptimo que los métodos clásicos.

## 4. Guía de respuestas a las preguntas de entrega

**1. Diferencia entre decisión de 1ª y 2ª etapa.**
Las de **primera etapa** se toman *antes* de observar la incertidumbre (aquí: qué unidades comprometer/encender/apagar) y son únicas para todo el sistema. Las de **segunda etapa** se toman *después* de revelarse el escenario (el despacho) y pueden adaptarse a cada realización $s$. La primera etapa es **anticipativa**; la segunda es **adaptativa (recurso)**.

**2. Qué variables son de cada etapa.**
- 1ª etapa: $u_{g,t}$ (estado on/off), $v_{g,t}$ (partida), $w_{g,t}$ (apagado). Sin índice $s$.
- 2ª etapa: $p_{g,t,s}$ (despacho convencional), $p^{sol}_{t,s}$ (solar usada), $\ell_{t,s}$ (energía no suministrada). Con índice $s$.

**3. Por qué las binarias de compromiso no deben depender del escenario.**
Porque la decisión de encender/apagar una unidad se materializa **antes** de saber qué escenario ocurrirá; físicamente no se puede tener "una central encendida en s1 y apagada en s3" al mismo tiempo. Imponer un único $u_{g,t}$ para todos los escenarios es la **restricción de no-anticipatividad**: obliga a una sola decisión robusta frente a toda la distribución, evitando una solución tramposa "clarividente" (perfect information).

**4. Cómo se incorpora la incertidumbre.**
Mediante un conjunto finito de escenarios $\mathcal S$, cada uno con perfiles propios $D_{t,s}$, $P^{sol}_{t,s}$ y probabilidad $\pi_s$. La incertidumbre entra en la función objetivo como **costo esperado de operación** $\sum_s \pi_s Q(x,\xi_s)$ y en las restricciones de balance/disponibilidad, que se replican **por cada escenario**.

**5. Determinista vs estocástica (conceptual).**
La determinista usa un único pronóstico (a menudo el promedio) y entrega una solución que es óptima *solo si* ese pronóstico se cumple; tiende a **sub-comprometer** capacidad y falla ante escenarios adversos. La estocástica optimiza el costo **esperado** sobre todos los escenarios manteniendo un compromiso común, por lo que **cubre (hedge)** las realizaciones malas. El **VSS > 0** cuantifica esa ventaja: la decisión basada en promedios (EEV), evaluada en la realidad incierta, resulta mucho más cara que la decisión estocástica (RP).

**6. Análisis de los resultados del notebook.**
- *Patrón de encendido:* la unidad base barata (G1) permanece encendida casi todo el horizonte; las unidades caras (G2, G3) se comprometen en horas punta. El modelo estocástico enciende capacidad **extra** respecto al EV para cubrir el escenario s3.
- *Uso de solar:* se aprovecha al máximo la solar disponible en horas diurnas (desplaza generación cara). En s3 (nublado) la solar cae y debe reemplazarse con convencional.
- *Energía no suministrada:* aparece **solo en el escenario crítico s3** y en las horas de mayor demanda, cuando la capacidad convencional + solar no alcanza. En s1 y s2 no hay ENS.

**7. Qué ocurre si se aumentan los escenarios.**
Mejora la **representación de la distribución** de la incertidumbre y suele dar decisiones de primera etapa más robustas y un costo esperado más confiable, pero **crece el tamaño del MILP** (más variables/restricciones de 2ª etapa) y por tanto el tiempo de cómputo. En el límite conviene usar **reducción de escenarios** o descomposición (p. ej. Benders / *progressive hedging*) para mantener el problema tratable.

---
*Notebook generado como guía de desarrollo de la Actividad 1 A2IC. Ajuste libremente datos, número de unidades, períodos y escenarios para experimentar.*
